In [ ]:
import numpy as np
import pandas as pd


"""
TrainTestDataSplit.ipynb

Take in the output from DataProcessing(surface_all.parquet)
1. Randomly hold out 20% of the data every day as test set 
2. Write to a pickle file

Loosely procedure from NN smoothing no-arbitrage paper. 

"""

DRIVE="/content/drive/MyDrive/Colab Notebooks/OptionsData/"
df = pd.read_parquet(f"{DRIVE}/OptionsData/processed/surface_all.parquet")     # the full panel from 01_data_prep

TICKER = "SPX"
df = df[df["ticker"] == TICKER].copy().reset_index(drop=True)
print(f"{TICKER}: {len(df)} quotes over {df['quote_date'].nunique()} days")

# hold out 20% of the set for training 
HOLDOUT_FRAC, MIN_VISIBLE, SPLIT_SEED = 0.20, 6, 0


def make_holdout(frame, frac, min_vis, seed):
    rng = np.random.default_rng(seed)
    hold = pd.Series(False, index=frame.index)
    for _, sl in frame.groupby(["ticker", "quote_date", "expire_date"]):
        n = len(sl)
        n_hold = min(int(round(frac * n)), max(0, n - min_vis))
        if n_hold > 0:
            # Randomly select the 20% for hold out. 
            hold.loc[rng.choice(sl.index.values, size=n_hold, replace=False)] = True
    return hold

df["is_holdout"] = make_holdout(df, HOLDOUT_FRAC, MIN_VISIBLE, SPLIT_SEED)

lvl = (df[~df["is_holdout"]].groupby(["ticker", "quote_date"])["iv"].median()
       .rename("day_level").reset_index())
df = df.merge(lvl, on=["ticker", "quote_date"], how="left")
df["day_level"] = df["day_level"].fillna(df["iv"].median())

n_vis, n_hold = (~df["is_holdout"]).sum(), df["is_holdout"].sum()
print(f"visible {n_vis}  held-out {n_hold}  ({100*n_hold/len(df):.1f}%)")


# Saving the file by pickling 
df.to_pickle(f"{DRIVE}/df_{TICKER}.pkl")           